In [0]:
from pyspark.sql import functions as F

customer_features_ml = spark.table(
    "workspace.kkbox.customer_features_ml"
)

model_comparison = spark.table(
    "workspace.kkbox.model_comparison"
)

print("Business insight tables loaded successfully.")

print("Customer rows:", customer_features_ml.count())
print("Model comparison rows:", model_comparison.count())

In [0]:
customer_features_ml = spark.table("workspace.kkbox.customer_features_ml")

print("Table loaded successfully")
print(customer_features_ml)

In [0]:
print("Customer rows:", customer_features_ml.count())

In [0]:
churn_distribution = (
    customer_features_ml
    .groupBy("is_churn")
    .count()
    .orderBy("is_churn")
)

display(churn_distribution)

In [0]:
from pyspark.sql import functions as F

In [0]:
churn_behavior = (
    customer_features_ml
    .groupBy("is_churn")
    .agg(
        F.round(F.avg("transaction_count"), 2)
         .alias("avg_transaction_count"),

        F.round(F.avg("total_amount_paid"), 2)
         .alias("avg_total_amount_paid"),

        F.round(F.avg("avg_amount_paid"), 2)
         .alias("avg_amount_paid"),

        F.round(F.avg("avg_plan_days"), 2)
         .alias("avg_plan_days"),

        F.round(
            F.avg(
                F.col("auto_renew_count") /
                F.col("transaction_count")
            ) * 100,
            2
        ).alias("auto_renew_rate_pct"),

        F.round(
            F.avg(
                F.col("cancel_count") /
                F.col("transaction_count")
            ) * 100,
            2
        ).alias("cancel_rate_pct"),

        F.round(
            F.avg(
                F.datediff(
                    F.col("latest_expiry_date"),
                    F.col("latest_transaction_date")
                )
            ),
            2
        ).alias("avg_days_from_last_transaction_to_expiry")
    )
    .orderBy("is_churn")
)

display(churn_behavior)

In [0]:
auto_renew_segments = (
    customer_features_ml
    .withColumn(
        "auto_renew_rate",
        F.col("auto_renew_count") / F.col("transaction_count")
    )
    .withColumn(
        "auto_renew_segment",
        F.when(F.col("auto_renew_rate") == 0, "No Auto-Renew")
         .when(F.col("auto_renew_rate") < 0.5, "Low Auto-Renew")
         .when(F.col("auto_renew_rate") < 1.0, "Mixed Auto-Renew")
         .otherwise("Always Auto-Renew")
    )
    .groupBy("auto_renew_segment")
    .agg(
        F.count("*").alias("customer_count"),
        F.sum("is_churn").alias("churned_customers"),
        F.round(F.avg("is_churn") * 100, 2).alias("churn_rate_pct")
    )
    .orderBy(F.desc("churn_rate_pct"))
)

display(auto_renew_segments)

In [0]:
cancel_segments = (
    customer_features_ml
    .withColumn(
        "cancel_rate",
        F.col("cancel_count") / F.col("transaction_count")
    )
    .withColumn(
        "cancel_segment",
        F.when(F.col("cancel_rate") == 0, "No Cancellations")
         .when(F.col("cancel_rate") < 0.25, "Low Cancellation")
         .when(F.col("cancel_rate") < 0.50, "Moderate Cancellation")
         .otherwise("High Cancellation")
    )
    .groupBy("cancel_segment")
    .agg(
        F.count("*").alias("customer_count"),
        F.sum("is_churn").alias("churned_customers"),
        F.round(F.avg("is_churn") * 100, 2).alias("churn_rate_pct")
    )
    .orderBy(F.desc("churn_rate_pct"))
)

display(cancel_segments)

In [0]:
high_risk_segment = (
    customer_features_ml
    .withColumn(
        "risk_segment",
        F.when(
            (F.col("auto_renew_count") == 0) &
            (F.col("cancel_count") > 0),
            "No Auto-Renew + Cancellation"
        )
        .when(
            F.col("auto_renew_count") == 0,
            "No Auto-Renew Only"
        )
        .when(
            F.col("cancel_count") > 0,
            "Cancellation Only"
        )
        .otherwise("Neither Signal")
    )
    .groupBy("risk_segment")
    .agg(
        F.count("*").alias("customer_count"),
        F.sum("is_churn").alias("churned_customers"),
        F.round(F.avg("is_churn") * 100, 2).alias("churn_rate_pct")
    )
    .orderBy(F.desc("churn_rate_pct"))
)

display(high_risk_segment)

In [0]:
signal_combinations = (
    customer_features_ml
    .withColumn(
        "auto_renew_status",
        F.when(F.col("auto_renew_count") == 0, "No Auto-Renew")
         .otherwise("Has Auto-Renew")
    )
    .withColumn(
        "cancellation_status",
        F.when(F.col("cancel_count") == 0, "No Cancellation")
         .otherwise("Has Cancellation")
    )
    .groupBy("auto_renew_status", "cancellation_status")
    .agg(
        F.count("*").alias("customer_count"),
        F.sum("is_churn").alias("churned_customers"),
        F.round(F.avg("is_churn") * 100, 2).alias("churn_rate_pct")
    )
    .orderBy("auto_renew_status", "cancellation_status")
)

display(signal_combinations)

In [0]:
from pyspark.ml.classification import RandomForestClassificationModel

model_path = "/Volumes/workspace/kkbox/models/churn_random_forest"

rf_model = RandomForestClassificationModel.load(model_path)

print("Random Forest loaded successfully.")
print("Number of trees:", rf_model.getNumTrees)

In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

# Features used by the churn model
categorical_cols = [
    "city",
    "gender",
    "registered_via"
]

numeric_cols = [
    "age",
    "transaction_count",
    "total_amount_paid",
    "avg_amount_paid",
    "total_plan_price",
    "avg_plan_days",
    "auto_renew_count",
    "cancel_count",
    "registration_year",
    "registration_month",
    "days_since_last_transaction"
]

indexers = [
    StringIndexer(
        inputCol=col,
        outputCol=f"{col}_indexed",
        handleInvalid="keep"
    )
    for col in categorical_cols
]

encoders = [
    OneHotEncoder(
        inputCol=f"{col}_indexed",
        outputCol=f"{col}_encoded"
    )
    for col in categorical_cols
]

assembler = VectorAssembler(
    inputCols=[
        *numeric_cols,
        *[f"{col}_encoded" for col in categorical_cols]
    ],
    outputCol="features",
    handleInvalid="keep"
)

feature_pipeline = Pipeline(
    stages=[*indexers, *encoders, assembler]
)

print("Feature pipeline created successfully.")

In [0]:
print("Customer feature columns:")
print(customer_features_ml.columns)

In [0]:
from pyspark.sql import functions as F

scoring_data = (
    customer_features_ml
    .withColumn(
        "registration_year",
        F.when(
            F.col("registration_date") != "unknown",
            F.year(F.to_date("registration_date"))
        ).otherwise(-1)
    )
    .withColumn(
        "registration_month",
        F.when(
            F.col("registration_date") != "unknown",
            F.month(F.to_date("registration_date"))
        ).otherwise(-1)
    )
    .withColumn(
        "days_since_last_transaction",
        F.datediff(
            F.col("latest_expiry_date"),
            F.col("latest_transaction_date")
        )
    )
)

print("Model-ready features created.")
print("Rows:", scoring_data.count())

In [0]:
feature_pipeline_model = feature_pipeline.fit(scoring_data)

scoring_prepared = feature_pipeline_model.transform(scoring_data)

print("Feature transformation completed.")
print("Scored rows:", scoring_prepared.count())

In [0]:
predictions = rf_model.transform(scoring_prepared)

print("Predictions generated successfully.")
print("Prediction rows:", predictions.count())

In [0]:
print("Model feature count:", rf_model.numFeatures)
print("Model feature column:", rf_model.getFeaturesCol())
print("Prepared feature vector size:", scoring_prepared.select("features").first()["features"].size)

In [0]:
# Recreate the derived features used during model training

scoring_data = (
    customer_features_ml
    .withColumn(
        "auto_renew_rate",
        F.col("auto_renew_count") / F.col("transaction_count")
    )
    .withColumn(
        "cancel_rate",
        F.col("cancel_count") / F.col("transaction_count")
    )
    .withColumn(
        "registration_year",
        F.when(
            F.col("registration_date") != "unknown",
            F.year(F.to_date("registration_date"))
        ).otherwise(-1)
    )
    .withColumn(
        "registration_month",
        F.when(
            F.col("registration_date") != "unknown",
            F.month(F.to_date("registration_date"))
        ).otherwise(-1)
    )
    .withColumn(
        "days_since_last_transaction",
        F.datediff(
            F.col("latest_expiry_date"),
            F.col("latest_transaction_date")
        )
    )
)

numeric_cols = [
    "age",
    "has_member_data",
    "transaction_count",
    "total_amount_paid",
    "avg_amount_paid",
    "total_plan_price",
    "avg_plan_days",
    "auto_renew_count",
    "cancel_count",
    "auto_renew_rate",
    "cancel_rate",
    "registration_year",
    "registration_month",
    "days_since_last_transaction"
]

categorical_cols = [
    "city",
    "gender",
    "registered_via"
]

indexers = [
    StringIndexer(
        inputCol=col,
        outputCol=f"{col}_indexed",
        handleInvalid="keep"
    )
    for col in categorical_cols
]

encoders = [
    OneHotEncoder(
        inputCol=f"{col}_indexed",
        outputCol=f"{col}_encoded"
    )
    for col in categorical_cols
]

assembler = VectorAssembler(
    inputCols=[
        *numeric_cols,
        *[f"{col}_encoded" for col in categorical_cols]
    ],
    outputCol="features",
    handleInvalid="keep"
)

feature_pipeline = Pipeline(
    stages=[*indexers, *encoders, assembler]
)

feature_pipeline_model = feature_pipeline.fit(scoring_data)

scoring_prepared = feature_pipeline_model.transform(scoring_data)

feature_size = scoring_prepared.select("features").first()["features"].size

print("Rebuilt feature pipeline successfully.")
print("Feature vector size:", feature_size)

In [0]:
predictions = rf_model.transform(scoring_prepared)

print("Corrected predictions generated successfully.")
print("Prediction rows:", predictions.count())

In [0]:
from pyspark.ml.functions import vector_to_array

customer_risk_scores = (
    predictions
    .select(
        "msno",
        "is_churn",
        vector_to_array("probability")[1].alias("churn_probability"),
        F.col("prediction").cast("int").alias("predicted_churn")
    )
)

print("Customer risk scores created.")
print("Rows:", customer_risk_scores.count())

In [0]:
customer_risk_segments = (
    customer_risk_scores
    .withColumn(
        "risk_segment",
        F.when(
            F.col("churn_probability") >= 0.75,
            "High Risk"
        )
        .when(
            F.col("churn_probability") >= 0.50,
            "Medium Risk"
        )
        .otherwise("Low Risk")
    )
)

risk_summary = (
    customer_risk_segments
    .groupBy("risk_segment")
    .agg(
        F.count("*").alias("customer_count"),
        F.sum("is_churn").alias("actual_churned_customers"),
        F.round(
            F.avg("is_churn") * 100,
            2
        ).alias("actual_churn_rate_pct"),
        F.round(
            F.avg("churn_probability") * 100,
            2
        ).alias("avg_predicted_churn_probability_pct")
    )
    .orderBy(
        F.desc("avg_predicted_churn_probability_pct")
    )
)

display(risk_summary)

In [0]:
# Recreate the same 80/20 split used during model training

train_df, test_df = scoring_data.randomSplit(
    [0.8, 0.2],
    seed=42
)

print("Training rows:", train_df.count())
print("Test rows:", test_df.count())

In [0]:
# Fit preprocessing only on the training split
training_feature_pipeline_model = feature_pipeline.fit(train_df)

# Apply the exact training mappings to all customers
scoring_prepared = training_feature_pipeline_model.transform(scoring_data)

feature_size = (
    scoring_prepared
    .select("features")
    .first()["features"]
    .size
)

print("Training-based preprocessing completed.")
print("Feature vector size:", feature_size)
print("Scoring rows:", scoring_prepared.count())

In [0]:
predictions = rf_model.transform(scoring_prepared)

print("Training-aligned predictions generated.")
print("Prediction rows:", predictions.count())

In [0]:
probability_check = (
    predictions
    .select(
        "is_churn",
        vector_to_array("probability")[1].alias("churn_probability")
    )
    .groupBy("is_churn")
    .agg(
        F.count("*").alias("customer_count"),
        F.round(
            F.avg("churn_probability") * 100,
            2
        ).alias("avg_predicted_churn_probability_pct"),
        F.round(
            F.expr("percentile(churn_probability, 0.5)") * 100,
            2
        ).alias("median_predicted_churn_probability_pct")
    )
    .orderBy("is_churn")
)

display(probability_check)

In [0]:
churn_feature_importance = spark.table(
    "workspace.kkbox.churn_feature_importance"
)

display(
    churn_feature_importance
    .orderBy(F.desc("importance"))
    .limit(15)
)

In [0]:
business_insights = spark.createDataFrame([
    (
        "Auto-Renewal",
        "Customers with no auto-renewal had a 31.92% churn rate.",
        "Prioritize customers without auto-renewal for renewal reminders and retention campaigns."
    ),
    (
        "Cancellation",
        "Customers in the high-cancellation segment had an 82.37% churn rate.",
        "Treat repeated cancellation activity as an early retention warning signal."
    ),
    (
        "Renewal and Cancellation",
        "Customers with auto-renewal and no cancellations had a 0.60% churn rate.",
        "Maintain strong renewal experiences and protect this low-risk customer segment."
    ),
    (
        "Model Drivers",
        "Days Since Last Transaction, Auto-Renew Rate, and Cancel Rate were the top three Random Forest features.",
        "Use recent activity and renewal behavior as key signals in churn monitoring."
    )
], [
    "insight_area",
    "finding",
    "business_action"
])

display(business_insights)

In [0]:
print("=== Notebook 10 Business Insights Validation ===")

print("Customer scope:", customer_features_ml.count())

print(
    "Overall churn rate:",
    round(
        customer_features_ml
        .select(F.avg("is_churn"))
        .first()[0] * 100,
        2
    ),
    "%"
)

print(
    "Business insights:",
    business_insights.count()
)

print(
    "Top model driver:",
    churn_feature_importance
    .orderBy(F.desc("importance"))
    .first()["feature_name"]
)

print("Notebook 10 validation completed successfully.")